***Подготовка***

In [5]:
!pip install -q dash

In [6]:
import gzip
import re
from collections import Counter
from datetime import timedelta

import pandas as pd
import plotly.express as px

from dash import Dash, html, dcc, Input, Output
from google.colab import output as colab_output

## А0. Грузим и фильтруем данные

In [7]:
REVIEWS_PATH = 'reviews_Toys_and_Games_5.json.gz'
META_PATH = 'meta_Toys_and_Games.json.gz'

PIECES_RE = re.compile(r'(\d+)[\s-]*pieces?', re.IGNORECASE)

def has_puzzles(cats):
    # рекурсивно ищем Puzzles на любом уровне
    if isinstance(cats, str):
        return cats == 'Puzzles'
    if isinstance(cats, list):
        return any(has_puzzles(x) for x in cats)
    return False

def category_leaf(cats):
    # самый глубокий элемент
    if not cats:
        return None
    if isinstance(cats, list) and cats:
        last = cats[-1]
        if isinstance(last, list) and last:
            return last[-1]
        return last
    return cats

def to_float(x):
    try:
        return float(x)
    except:
        return float('nan')

def extract_pieces(title):
    if not title:
        return None
    m = PIECES_RE.search(title)
    return int(m.group(1)) if m else None

In [8]:
# мета: 336к товаров, сразу фильтруем пазлы
meta_rows = []
with gzip.open(META_PATH, 'r') as g:
    for line in g:
        try:
            d = eval(line)
        except:
            continue
        if has_puzzles(d.get('categories', [])):
            meta_rows.append({
                'asin': d.get('asin'),
                'title': d.get('title'),
                'brand': d.get('brand'),
                'price': to_float(d.get('price')),
                'category_leaf': category_leaf(d.get('categories', [])),
            })

meta = pd.DataFrame(meta_rows)
print('товаров пазлов:', len(meta))
meta.head()

товаров пазлов: 14427


,asin,title,brand,price,category_leaf
0,0000191639,Dr. Suess 19163 Dr. Seuss Puzzle 3 Pack Bundle,Dr. Seuss,37.12,Jigsaw Puzzles
1,0375829695,Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,Dr. Seuss,24.82,Jigsaw Puzzles
2,0439400066,3D Puzzle Buster,None,99.15,3-D Puzzles
3,0439845297,Insect Model Lytta Stygica Green Flower Beetle...,None,8.99,3-D Puzzles
4,0545302285,Scholastic Teacher's Friend Rhyming Learning P...,Scholastic Teacher&#39;s Friend,8.94,Jigsaw Puzzles


In [9]:
# отзывы: оставляем только по нашим asin
asins = set(meta['asin'].dropna())
review_rows = []
with gzip.open(REVIEWS_PATH, 'r') as g:
    for line in g:
        try:
            d = eval(line)
        except:
            continue
        if d.get('asin') in asins:
            review_rows.append({
                'reviewerID': d.get('reviewerID'),
                'reviewerName': d.get('reviewerName'),
                'asin': d.get('asin'),
                'overall': d.get('overall'),
                'summary': d.get('summary'),
                'unixReviewTime': d.get('unixReviewTime'),
            })

reviews = pd.DataFrame(review_rows)
print('отзывов на пазлы:', len(reviews))
reviews.head()

отзывов на пазлы: 6998


,reviewerID,reviewerName,asin,overall,summary,unixReviewTime
0,ABKGN0ETNUNOA,Beth S,073533305X,5.0,Adorable puzzle,1391731200
1,A3FFY8SLBK89ES,"B. Rachal ""msbrachal""",073533305X,5.0,great puzzle,1390262400
2,A2CDP7ULSHS5G9,BuyerG,073533305X,5.0,Developmentally Appropriate - Develops great s...,1388188800
3,ASSHPKQBFZPB5,"M. C. ""lovestoread""",073533305X,5.0,Great,1384473600
4,A3U6SWQW5ID7F5,PNaz,073533305X,5.0,We love this puzzle!,1390867200


In [10]:
# джойн
df = reviews.merge(meta, on='asin', how='inner')
df['review_date'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['pieces'] = df['title'].apply(extract_pieces)
print('строк после джойна:', len(df))
df.head()

строк после джойна: 6998


,reviewerID,reviewerName,asin,overall,summary,unixReviewTime,title,brand,price,category_leaf,review_date,pieces
0,ABKGN0ETNUNOA,Beth S,073533305X,5.0,Adorable puzzle,1391731200,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-02-07,NaN
1,A3FFY8SLBK89ES,"B. Rachal ""msbrachal""",073533305X,5.0,great puzzle,1390262400,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-01-21,NaN
2,A2CDP7ULSHS5G9,BuyerG,073533305X,5.0,Developmentally Appropriate - Develops great s...,1388188800,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2013-12-28,NaN
3,ASSHPKQBFZPB5,"M. C. ""lovestoread""",073533305X,5.0,Great,1384473600,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2013-11-15,NaN
4,A3U6SWQW5ID7F5,PNaz,073533305X,5.0,We love this puzzle!,1390867200,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-01-28,NaN


In [11]:
# таблица юзеров (уникальные reviewerID)
users = df.groupby('reviewerID').agg(
    reviewer_name=('reviewerName', 'first'),
    first_review_date=('review_date', 'min'),
    n_reviews=('reviewerID', 'size'),
).reset_index()
print('юзеров:', len(users))
users.head()

юзеров: 3901


,reviewerID,reviewer_name,first_review_date,n_reviews
0,A09027871HUIYHTFHPJX6,Leslie Hawkins,2013-03-11,1
1,A101C99CG8EFUH,Benjamin McGough,2011-03-28,1
2,A102D49JO9BOZY,Julie O.,2013-01-15,1
3,A1051DBTLWP5A2,Monkey Momma,2012-01-11,1
4,A105LFG3MT4N23,TM,2011-06-26,1


## А1. Класс UserProfile

In [12]:
class UserProfile:

    def __init__(self, reviewer_id, name, registration_date, n_reviews):
        self.reviewer_id = reviewer_id
        self.name = None if pd.isna(name) else name
        self.registration_date = pd.to_datetime(registration_date)
        self.n_reviews = int(n_reviews)

    @property
    def visit_frequency(self):
        # отзывов в день с момента первого отзыва
        days = (pd.Timestamp.now() - self.registration_date).days
        if days <= 0:
            return 0.0
        return self.n_reviews / days

    @classmethod
    def from_row(cls, row):
        return cls(
            reviewer_id=row['reviewerID'],
            name=row['reviewer_name'],
            registration_date=row['first_review_date'],
            n_reviews=row['n_reviews'],
        )

    def __repr__(self):
        return f'UserProfile(id={self.reviewer_id!r}, name={self.name!r}, n_reviews={self.n_reviews})'

    def __eq__(self, other):
        if not isinstance(other, UserProfile):
            return NotImplemented
        return self.reviewer_id == other.reviewer_id

    def __hash__(self):
        return hash(self.reviewer_id)

In [13]:
# 5 профилей из первых строк
for _, row in users.head(5).iterrows():
    u = UserProfile.from_row(row)
    print(u)
    print(f'  registration={u.registration_date.date()} | visit_frequency={round(u.visit_frequency, 5)}')

# проверка __eq__ и __hash__
u1 = UserProfile.from_row(users.iloc[0])
u2 = UserProfile.from_row(users.iloc[0])
print()
print('равны:', u1 == u2, '| hash одинаковый:', hash(u1) == hash(u2))

UserProfile(id='A09027871HUIYHTFHPJX6', name='Leslie Hawkins', n_reviews=1)
  registration=2013-03-11 | visit_frequency=0.00021
UserProfile(id='A101C99CG8EFUH', name='Benjamin McGough', n_reviews=1)
  registration=2011-03-28 | visit_frequency=0.00018
UserProfile(id='A102D49JO9BOZY', name='Julie O.', n_reviews=1)
  registration=2013-01-15 | visit_frequency=0.00021
UserProfile(id='A1051DBTLWP5A2', name='Monkey Momma', n_reviews=1)
  registration=2012-01-11 | visit_frequency=0.00019
UserProfile(id='A105LFG3MT4N23', name='TM', n_reviews=1)
  registration=2011-06-26 | visit_frequency=0.00018

равны: True | hash одинаковый: True


## А2. Класс Order


In [14]:
class Order:

    def __init__(self, reviewer_id, asin, date, price, title=None, brand=None,
                 category=None, pieces=None, rating=None, summary=None):
        self.reviewer_id = reviewer_id
        self.asin = asin
        self.date = pd.to_datetime(date)
        self.price = float(price) if not pd.isna(price) else None
        self.title = None if pd.isna(title) else title
        self.brand = None if pd.isna(brand) else brand
        self.category = None if pd.isna(category) else category
        self.pieces = int(pieces) if not pd.isna(pieces) else None
        self.rating = float(rating) if not pd.isna(rating) else None
        self.summary = None if pd.isna(summary) else summary

    @property
    def total(self):
        # без скидок и количества - просто цена
        return self.price

    def user(self, registry=None):
        # если передали словарь {id: UserProfile} - вернем профиль, иначе id
        if registry is not None:
            return registry.get(self.reviewer_id)
        return self.reviewer_id

    @classmethod
    def from_row(cls, row):
        return cls(
            reviewer_id=row['reviewerID'],
            asin=row['asin'],
            date=row['review_date'],
            price=row.get('price'),
            title=row.get('title'),
            brand=row.get('brand'),
            category=row.get('category_leaf'),
            pieces=row.get('pieces'),
            rating=row.get('overall'),
            summary=row.get('summary'),
        )

    def __repr__(self):
        d = self.date.date() if pd.notna(self.date) else None
        return f'Order(user={self.reviewer_id!r}, asin={self.asin!r}, date={d}, total={self.total})'

    def __eq__(self, other):
        if not isinstance(other, Order):
            return NotImplemented
        return (self.reviewer_id == other.reviewer_id
                and self.asin == other.asin
                and self.date == other.date)

    def __hash__(self):
        return hash((self.reviewer_id, self.asin, self.date))

In [15]:
# 5 заказов
for _, row in df.head(5).iterrows():
    o = Order.from_row(row)
    print(o)
    print(f'  category={o.category} | brand={o.brand} | rating={o.rating} | pieces={o.pieces}')


print()
registry = {row['reviewerID']: UserProfile.from_row(row) for _, row in users.head(100).iterrows()}
sample_row = df[df['reviewerID'].isin(registry)].iloc[0]
o = Order.from_row(sample_row)
print('o.user(registry):', o.user(registry))
print('o.user() без registry:', o.user())

Order(user='ABKGN0ETNUNOA', asin='073533305X', date=2014-02-07, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A3FFY8SLBK89ES', asin='073533305X', date=2014-01-21, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A2CDP7ULSHS5G9', asin='073533305X', date=2013-12-28, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='ASSHPKQBFZPB5', asin='073533305X', date=2013-11-15, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A3U6SWQW5ID7F5', asin='073533305X', date=2014-01-28, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None

o.user(registry): UserProfile(id='A10E3F50DIUJEE', name='C Wahlman "cdub"', n_reviews=5)
o.user() без registry: A10E3F50DIUJEE


## А3. Класс UserAnalytics

Метрики по одному юзеру. Принимает UserProfile + список Order этого юзера.

Считаем:
- **LTV** - сумма total по всем заказам
- **AOV** - средний чек
- **churn** - True если последний заказ > 90 дней назад
- **retention_30/60/90** - вернулся ли в окно X дней после первого заказа
- **rfm_segment** - Champion / Loyal / Promising / At Risk / Lost (по скорам R, F, M по 1-5)
- **favorite_category** - чаще всего покупаемая категория
- **complexity_preference** - средний размер пазла + ярлык (легкие/средние/сложные)


In [16]:
class UserAnalytics:

    # точка отсчета для recency. По умолчанию реальное сейчас.
    REF_DATE = pd.Timestamp.now()

    def __init__(self, user, orders):
        self._user = user
        self._orders = list(orders)

    @property
    def user(self):
        return self._user

    @property
    def orders(self):
        return self._orders

    @property
    def ltv(self):
        return sum(o.total for o in self._orders if o.total is not None)

    @property
    def aov(self):
        if not self._orders:
            return 0.0
        return self.ltv / len(self._orders)

    @property
    def first_order_date(self):
        dates = [o.date for o in self._orders if pd.notna(o.date)]
        return min(dates) if dates else None

    @property
    def last_order_date(self):
        dates = [o.date for o in self._orders if pd.notna(o.date)]
        return max(dates) if dates else None

    @property
    def churn(self):
        last = self.last_order_date
        if last is None:
            return True
        return (self.REF_DATE - last).days > 90

    def _retention_window(self, days):
        # вернулся ли в окно X дней после первого заказа
        first = self.first_order_date
        if first is None:
            return False
        threshold = first + timedelta(days=days)
        return any(first < o.date <= threshold for o in self._orders if pd.notna(o.date))

    @property
    def retention_30(self):
        return self._retention_window(30)

    @property
    def retention_60(self):
        return self._retention_window(60)

    @property
    def retention_90(self):
        return self._retention_window(90)

    def _rfm_scores(self):
        # R - давность последнего заказа
        last = self.last_order_date
        if last is None:
            r = 1
        else:
            days = (self.REF_DATE - last).days
            if days < 30:
                r = 5
            elif days < 90:
                r = 4
            elif days < 180:
                r = 3
            elif days < 365:
                r = 2
            else:
                r = 1

        # F - число заказов
        n = len(self._orders)
        if n >= 10:
            f = 5
        elif n >= 5:
            f = 4
        elif n >= 3:
            f = 3
        elif n == 2:
            f = 2
        else:
            f = 1

        # M - выручка
        ltv = self.ltv
        if ltv >= 500:
            m = 5
        elif ltv >= 200:
            m = 4
        elif ltv >= 100:
            m = 3
        elif ltv >= 50:
            m = 2
        else:
            m = 1

        return r, f, m

    @property
    def rfm_segment(self):
        r, f, m = self._rfm_scores()
        if r >= 4 and f >= 4 and m >= 4:
            return 'Champion'
        if r >= 3 and f >= 3:
            return 'Loyal'
        if r >= 4 and f <= 2:
            return 'Promising'
        if r <= 2 and f >= 3:
            return 'At Risk'
        return 'Lost'

    @property
    def favorite_category(self):
        cats = [o.category for o in self._orders if o.category is not None]
        if not cats:
            return None
        return Counter(cats).most_common(1)[0][0]

    @property
    def complexity_preference(self):
        pcs = [o.pieces for o in self._orders if o.pieces is not None]
        if not pcs:
            return None
        avg = sum(pcs) / len(pcs)
        if avg < 100:
            level = 'легкие'
        elif avg < 500:
            level = 'средние'
        else:
            level = 'сложные'
        return {'avg_pieces': round(avg, 1), 'level': level}

    def __len__(self):
        return len(self._orders)

    def __iter__(self):
        return iter(self._orders)

    def __repr__(self):
        return (f'UserAnalytics(user={self._user.reviewer_id!r}, '
                f'orders={len(self._orders)}, ltv={self.ltv:.2f}, '
                f'segment={self.rfm_segment!r})')

In [17]:

UserAnalytics.REF_DATE = df['review_date'].max() + timedelta(days=60)
print('REF_DATE:', UserAnalytics.REF_DATE.date())

# берем самого активного юзера
counts = df.groupby('reviewerID').size().sort_values(ascending=False)
top_id = counts.index[0]
print('самый активный:', top_id, '| заказов:', counts.iloc[0])

user = UserProfile.from_row(users[users['reviewerID'] == top_id].iloc[0])
user_orders = [Order.from_row(r) for _, r in df[df['reviewerID'] == top_id].iterrows()]
a = UserAnalytics(user, user_orders)
a

REF_DATE: 2014-09-17
самый активный: A1EVV74UQYVKRY | заказов: 22


UserAnalytics(user='A1EVV74UQYVKRY', orders=22, ltv=402.56, segment='Champion')

In [18]:
# все метрики словарем
{
    'ltv': round(a.ltv, 2),
    'aov': round(a.aov, 2),
    'first_order': a.first_order_date.date(),
    'last_order': a.last_order_date.date(),
    'churn': a.churn,
    'retention_30/60/90': (a.retention_30, a.retention_60, a.retention_90),
    'rfm_scores (R, F, M)': a._rfm_scores(),
    'segment': a.rfm_segment,
    'favorite_category': a.favorite_category,
    'complexity': a.complexity_preference,
    'len(a)': len(a),
}

{'ltv': 402.56,
 'aov': 18.3,
 'first_order': datetime.date(2009, 1, 11),
 'last_order': datetime.date(2014, 7, 8),
 'churn': False,
 'retention_30/60/90': (False, False, False),
 'rfm_scores (R, F, M)': (4, 5, 4),
 'segment': 'Champion',
 'favorite_category': 'Jigsaw Puzzles',
 'complexity': {'avg_pieces': 928.0, 'level': 'сложные'},
 'len(a)': 22}

## А4. Класс BusinessDashboard + дашборд на dash


In [19]:
class BusinessDashboard:

    def __init__(self, analytics):
        self._analytics = list(analytics)

    @property
    def analytics(self):
        return self._analytics

    @property
    def total_revenue(self):
        return sum(a.ltv for a in self._analytics)

    @property
    def avg_ltv(self):
        if not self._analytics:
            return 0.0
        return self.total_revenue / len(self._analytics)

    @property
    def churn_rate(self):
        if not self._analytics:
            return 0.0
        return sum(1 for a in self._analytics if a.churn) / len(self._analytics)

    @property
    def rfm_breakdown(self):
        return dict(Counter(a.rfm_segment for a in self._analytics))

    def top_categories(self, n=10):
        cats = []
        for a in self._analytics:
            for o in a:
                if o.category is not None:
                    cats.append(o.category)
        return Counter(cats).most_common(n)

    def top_brands(self, n=10):
        brands = []
        for a in self._analytics:
            for o in a:
                if o.brand is not None:
                    brands.append(o.brand)
        return Counter(brands).most_common(n)

    @property
    def revenue_by_year(self):
        rev = {}
        for a in self._analytics:
            for o in a:
                if o.total is not None and pd.notna(o.date):
                    y = o.date.year
                    rev[y] = rev.get(y, 0) + o.total
        return dict(sorted(rev.items()))

    @property
    def revenue_by_category(self):
        rev = {}
        for a in self._analytics:
            for o in a:
                if o.total is not None and o.category is not None:
                    rev[o.category] = rev.get(o.category, 0) + o.total
        return rev

    def __len__(self):
        return len(self._analytics)

    def __repr__(self):
        return (f'BusinessDashboard(users={len(self._analytics)}, '
                f'revenue={self.total_revenue:.2f})')

In [20]:
# собираем профили + аналитику по всем юзерам
profiles_all = {row['reviewerID']: UserProfile.from_row(row) for _, row in users.iterrows()}

analytics_list = []
for rid, grp in df.groupby('reviewerID'):
    user_orders = [Order.from_row(r) for _, r in grp.iterrows()]
    analytics_list.append(UserAnalytics(profiles_all[rid], user_orders))

bd = BusinessDashboard(analytics_list)
bd

BusinessDashboard(users=3901, revenue=106374.10)

In [21]:
# базовые цифры
print(f'total revenue: ${bd.total_revenue:,.2f}')
print(f'avg LTV: ${bd.avg_ltv:,.2f}')
print(f'churn rate: {bd.churn_rate * 100:.1f}%')
print(f'юзеров: {len(bd):,}')
print()
print('RFM:', bd.rfm_breakdown)
print()
print('топ-10 категорий:')
for c, n in bd.top_categories(10):
    print(f'  {c:35s} -> {n}')
print()
print('топ-5 брендов:')
for b, n in bd.top_brands(5):
    print(f'  {b:35s} -> {n}')

total revenue: $106,374.10
avg LTV: $27.27
churn rate: 98.0%
юзеров: 3,901

RFM: {'Lost': 3135, 'At Risk': 658, 'Loyal': 44, 'Promising': 63, 'Champion': 1}

топ-10 категорий:
  Jigsaw Puzzles                      -> 3313
  Pegged Puzzles                      -> 1149
  Floor Puzzles                       -> 963
  Maze & Sequential Puzzles           -> 438
  3-D Puzzles                         -> 350
  Puzzles                             -> 299
  Puzzle Accessories                  -> 199
  Assembly & Disentanglement Puzzles  -> 172
  Puzzle Boxes                        -> 65
  Brain Teasers                       -> 40

топ-5 брендов:
  Melissa &amp; Doug                  -> 2610
  Ravensburger                        -> 2349
  PlaSmart                            -> 211
  Pastime Puzzles                     -> 174
  Think Fun                           -> 161


In [22]:
r = bd.rfm_breakdown
px.bar(x=list(r.keys()), y=list(r.values()),
       title='RFM сегменты', labels={'x': 'сегмент', 'y': 'юзеров'})

In [23]:
cats = bd.top_categories(10)
px.bar(x=[c[0] for c in cats], y=[c[1] for c in cats],
       title='Топ-10 категорий по числу заказов',
       labels={'x': 'категория', 'y': 'заказов'})

In [24]:
y = bd.revenue_by_year
px.bar(x=list(y.keys()), y=list(y.values()),
       title='Выручка по годам', labels={'x': 'год', 'y': 'выручка $'})

### Интерактивный дашборд на dash

In [25]:

# готовим фигуры
_r = bd.rfm_breakdown
fig_rfm = px.bar(x=list(_r.keys()), y=list(_r.values()),
                 title='RFM сегменты', labels={'x': 'сегмент', 'y': 'юзеров'})

_c = bd.top_categories(10)
fig_cat = px.bar(x=[c[0] for c in _c], y=[c[1] for c in _c],
                 title='Топ категорий', labels={'x': 'категория', 'y': 'заказов'})


def card(title, value):
    return html.Div([
        html.Div(title, style={'fontSize': 13, 'color': '#666'}),
        html.Div(value, style={'fontSize': 24, 'fontWeight': 'bold', 'marginTop': '4px'}),
    ], style={'padding': '15px', 'borderRadius': '8px', 'background': '#f5f5f5',
              'flex': 1, 'margin': '5px', 'textAlign': 'center'})


app = Dash()
app.layout = html.Div([
    html.H2('Пазлы - дашборд', style={'textAlign': 'center'}),
    html.Div([
        card('Общая выручка', f'${bd.total_revenue:,.0f}'),
        card('Avg LTV', f'${bd.avg_ltv:,.1f}'),
        card('Churn rate', f'{bd.churn_rate * 100:.1f}%'),
        card('Юзеров', f'{len(bd):,}'),
    ], style={'display': 'flex'}),
    html.Div([
        dcc.Graph(figure=fig_rfm, style={'flex': 1}),
        dcc.Graph(figure=fig_cat, style={'flex': 1}),
    ], style={'display': 'flex'}),
    html.Div([
        html.Label('Срез выручки: ', style={'marginRight': '10px'}),
        dcc.Dropdown(
            id='cut-dd',
            options=[
                {'label': 'по годам', 'value': 'year'},
                {'label': 'по категориям', 'value': 'cat'},
            ],
            value='year',
            style={'width': '300px'},
        ),
    ], style={'display': 'flex', 'alignItems': 'center', 'padding': '10px'}),
    dcc.Graph(id='cut-graph'),
], style={'fontFamily': 'sans-serif', 'maxWidth': '1200px', 'margin': '0 auto'})


@app.callback(Output('cut-graph', 'figure'), Input('cut-dd', 'value'))
def update_cut(val):
    if val == 'year':
        r = bd.revenue_by_year
        return px.bar(x=list(r.keys()), y=list(r.values()),
                      title='Выручка по годам',
                      labels={'x': 'год', 'y': 'выручка $'})
    r = bd.revenue_by_category
    return px.bar(x=list(r.keys()), y=list(r.values()),
                  title='Выручка по категориям',
                  labels={'x': 'категория', 'y': 'выручка $'})


# запускаем сервер в фоне и пробрасываем порт через прокси Colab
import threading
threading.Thread(target=lambda: app.run(port=8050, debug=False), daemon=True).start()

# даём серверу пару секунд подняться, потом показываем iframe
import time
time.sleep(2)
colab_output.serve_kernel_port_as_iframe(8050, height='900px')

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit


<IPython.core.display.Javascript object>

In [26]:
pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.7 MB/s eta 0:00:00


INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:56] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1779009430.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:56] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1779009430.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:56] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1779009430.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:57] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m1779009430.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:58] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_1_0m1779009430.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:58] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1779009430.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17

  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:58] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1779009430.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:59] "GET /_dash-component-suites/dash/dash_table/bundle.v7_1_0m1779009430.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:59] "GET /_dash-layout HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:22:59] "GET /_dash-dependencies HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:23:00] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:23:00] "GET /_dash-component-suites/plotly/package_data/plotly.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:23:00] "POST /_dash-update-component HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/May/2026 09:23:00] "GET /_dash-component-suites/dash/dcc/async-dropdown.js HTTP/1.1" 200 -


In [27]:
pip install pytrends requests faker matplotlib seaborn pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.6 MB/s eta 0:00:00


In [28]:
# --- импорты ---

import random
import math
from datetime import datetime, timedelta
from collections import defaultdict

import requests
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pytrends.request import TrendReq
from faker import Faker
import requests
from bs4 import BeautifulSoup
import pandas as pd
from typing import Dict, List, Optional
import time
import re
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse


# toysland


In [29]:


class ToysLandParser:

    def __init__(self, url: str, max_pages: int = None, delay: float = 1.0):
        self.base_url = url
        self.max_pages = max_pages
        self.delay = delay
        self.base_domain = "https://www.toys-land.ru"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        self.products = []
        self.total_pages_parsed = 0

    def fetch_page(self, url: str) -> str:
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            return ""

    def build_page_url(self, page_number: int) -> str:
        parsed_url = urlparse(self.base_url)
        query_params = parse_qs(parsed_url.query)
        query_params['page'] = [str(page_number)]
        new_query = urlencode(query_params, doseq=True)
        new_parsed_url = parsed_url._replace(query=new_query)
        return urlunparse(new_parsed_url)

    def get_total_pages(self, html: str) -> int:
        if not html:
            return 1

        soup = BeautifulSoup(html, 'html.parser')
        pagination = soup.find('div', class_='pagination') or soup.find('ul', class_='pagination') or soup.find('div', class_='nav-pages')

        if pagination:
            page_links = pagination.find_all('a')
            page_numbers = []

            for link in page_links:
                try:
                    page_num = int(link.text.strip())
                    page_numbers.append(page_num)
                except (ValueError, AttributeError):
                    continue

            if page_numbers:
                return max(page_numbers)

        products = soup.find_all('div', class_='name')
        if products:
            next_page_url = self.build_page_url(2)
            next_html = self.fetch_page(next_page_url)
            if next_html:
                next_products = BeautifulSoup(next_html, 'html.parser').find_all('div', class_='name')
                if next_products:
                    return self._find_total_pages_iteratively()

        return 1

    def _find_total_pages_iteratively(self) -> int:
        page = 2
        while True:
            if self.max_pages and page > self.max_pages:
                return self.max_pages

            url = self.build_page_url(page)
            html = self.fetch_page(url)

            if not html:
                return page - 1

            products = BeautifulSoup(html, 'html.parser').find_all('div', class_='name')
            if not products:
                return page - 1

            page += 1
            time.sleep(self.delay)

    def parse_products(self, html: str) -> List[Dict[str, str]]:
        if not html:
            return []

        soup = BeautifulSoup(html, 'html.parser')
        name_blocks = soup.find_all('div', class_='name')

        products = []
        for block in name_blocks:
            link = block.find('a', class_='hint')
            if link:
                href = link.get('href', '')
                full_url = self.base_domain + href if href.startswith('/') else href
                name_span = link.find('span', itemprop='name')
                name = name_span.text.strip() if name_span else ''

                product_info = {
                    'name': name,
                    'link': full_url,
                    'page': self.total_pages_parsed + 1
                }

                products.append(product_info)

        return products

    def parse_all_pages(self) -> List[Dict[str, str]]:
        all_products = []
        first_page_html = self.fetch_page(self.base_url)

        if not first_page_html:
            return []

        first_page_products = self.parse_products(first_page_html)
        all_products.extend(first_page_products)
        self.total_pages_parsed = 1

        total_pages = self.get_total_pages(first_page_html)

        if self.max_pages:
            total_pages = min(total_pages, self.max_pages)

        for page in range(2, total_pages + 1):
            time.sleep(self.delay)
            page_url = self.build_page_url(page)
            page_html = self.fetch_page(page_url)

            if not page_html:
                continue

            page_products = self.parse_products(page_html)

            if not page_products:
                break

            all_products.extend(page_products)
            self.total_pages_parsed = page

        self.products = all_products
        return all_products

    def create_dataframe(self) -> pd.DataFrame:
        if not self.products:
            return pd.DataFrame()

        df = pd.DataFrame(self.products)
        return df

    def run(self, use_pagination: bool = True) -> pd.DataFrame:
        if use_pagination:
            self.parse_all_pages()
        else:
            html = self.fetch_page(self.base_url)
            if html:
                self.products = self.parse_products(html)
                self.total_pages_parsed = 1
            else:
                return pd.DataFrame()

        df = self.create_dataframe()
        return df


url = "https://www.toys-land.ru/catalog/pazly/pazly-dlya-vzroslyh/?srsltid=AfmBOooesY5hQ_V-gzZeuZVTO3QpDrxsOvh_OfPxZUac-iBDIA-yGs9t"


parser = ToysLandParser(url, max_pages=5, delay=1.5)

df = parser.run(use_pagination=True)

display(df)

,name,link,page
0,"Пазл-репродукция Гончарова Н.С. - Подсолнухи, ...",https://www.toys-land.ru/goods/pazl-reprodukci...,1
1,"Пазл-репродукция Репин И.Е - На меже, 1500 эле...",https://www.toys-land.ru/goods/kartina-pazl-re...,1
2,"Пазл Германия - Ночь в Рамзау, 500 деталей",https://www.toys-land.ru/goods/pazl-germaniya-...,1
3,"Деревянный пазл Сальвадор Дали 40*40 см, 475 э...",https://www.toys-land.ru/goods/derevyannyj-paz...,1
4,"Пазл Парусная лодка, 1000 элементов",https://www.toys-land.ru/goods/pazl-parusnaya-...,1
...,...,...,...
88,"Пазл Парк Гуэль - Барселона, 500 элементов",https://www.toys-land.ru/goods/pazl-park-gujel...,2
89,"Пазл Цвета Азии, 1000 элементов",https://www.toys-land.ru/goods/pazl-cveta-azii...,2
90,Пазл-репродукция Жестокая преданность - Энн Ст...,https://www.toys-land.ru/goods/pazl-zhestokaya...,2
91,"Набор пазлов Загородный сад, 3*500 элементов",https://www.toys-land.ru/goods/nabor-pazlov-za...,2


In [30]:
class ProductParser:

    def __init__(self, delay: float = 0.5):
        self.delay = delay
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }

    def fetch_page(self, url: str) -> str:
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            return ""

    def parse_product_details(self, html: str, product_url: str) -> Dict[str, str]:
        if not html:
            return {}

        soup = BeautifulSoup(html, 'html.parser')

        product_info = {
            'url': product_url,
            'name': self._parse_product_name(soup),
            'manufacturer': '',
            'manufacturer_url': '',
            'country': '',
            'article': '',
            'price': '',
            'puzzle_size': '',
            'elements_count': '',
            'material': '',
            'packaging': '',
            'description': ''
        }

        self._parse_characteristics_table(soup, product_info)
        self._parse_characteristics_list(soup, product_info)
        self._parse_description(soup, product_info)

        return product_info

    def _parse_product_name(self, soup: BeautifulSoup) -> str:
        name_selectors = [
            ('h1', {}),
            ('div', {'class': 'product-name'}),
            ('span', {'itemprop': 'name'}),
            ('div', {'class': 'name'})
        ]

        for tag, attrs in name_selectors:
            element = soup.find(tag, attrs)
            if element:
                return element.text.strip()

        return ''

    def _parse_characteristics_table(self, soup: BeautifulSoup, product_info: Dict[str, str]):
        tables = soup.find_all('table')

        for table in tables:
            tbody = table.find('tbody')
            if not tbody:
                continue

            rows = tbody.find_all('tr')

            for row in rows:
                cells = row.find_all('td')
                if len(cells) >= 2:
                    label = cells[0].text.strip()
                    value_cell = cells[1]

                    if 'Изготовитель' in label or 'Производитель' in label:
                        link = value_cell.find('a')
                        if link:
                            product_info['manufacturer'] = link.text.strip()
                            product_info['manufacturer_url'] = link.get('href', '')
                        else:
                            product_info['manufacturer'] = value_cell.text.strip()

                    elif 'Страна' in label:
                        product_info['country'] = value_cell.text.strip()

                    elif 'Артикул' in label:
                        product_info['article'] = value_cell.text.strip()

                    elif 'Цена' in label or 'цена' in label.lower():
                        price_text = value_cell.text.strip()
                        product_info['price'] = self._clean_price(price_text)

    def _parse_characteristics_list(self, soup: BeautifulSoup, product_info: Dict[str, str]):
        lists = soup.find_all('ul')

        for ul in lists:
            items = ul.find_all('li')

            for item in items:
                text = item.text.strip()

                if 'Размер' in text and ('пазл' in text.lower() or 'собран' in text.lower()):
                    product_info['puzzle_size'] = self._extract_value(text, 'Размер')

                elif 'Количество элементов' in text or 'элементов' in text.lower():
                    product_info['elements_count'] = self._extract_value(text, 'элементов')

                elif 'Материал' in text:
                    product_info['material'] = self._extract_value(text, 'Материал')

                elif 'Упаковка' in text:
                    product_info['packaging'] = self._extract_value(text, 'Упаковка')

    def _parse_description(self, soup: BeautifulSoup, product_info: Dict[str, str]):
        description_selectors = [
            ('div', {'class': 'description'}),
            ('div', {'itemprop': 'description'}),
            ('div', {'class': 'product-description'}),
            ('div', {'id': 'description'})
        ]

        for tag, attrs in description_selectors:
            element = soup.find(tag, attrs)
            if element:
                product_info['description'] = element.text.strip()
                break

    def _extract_value(self, text: str, keyword: str) -> str:
        value = text.replace(keyword, '').strip()
        if value.startswith(':'):
            value = value[1:].strip()
        return value

    def _clean_price(self, price_text: str) -> str:
        price = price_text.strip()
        return price

    def parse_product(self, url: str) -> Dict[str, str]:
        html = self.fetch_page(url)

        if not html:
            return {}

        product_info = self.parse_product_details(html, url)
        time.sleep(self.delay)

        return product_info

    def parse_multiple_products(self, urls: List[str]) -> pd.DataFrame:
        all_products = []

        for i, url in enumerate(urls, 1):
            product_info = self.parse_product(url)

            if product_info:
                all_products.append(product_info)

            if i % 10 == 0:
                time.sleep(2)

        df = pd.DataFrame(all_products)
        return df


product_parser = ProductParser(delay=1.0)


urls = list(df["link"])
df_multiple = product_parser.parse_multiple_products(urls)

In [31]:
display(df_multiple)

,url,name,manufacturer,manufacturer_url,country,article,price,puzzle_size,elements_count,material,packaging,description
0,https://www.toys-land.ru/goods/pazl-reprodukci...,"Пазл-репродукция Гончарова Н.С. - Подсолнухи, ...",Стелла,/brand/stella,Россия,TG100106,940 ₽,собранного пазла: 68*48 см.,Количество : 1000 шт.,картон.,красочная картонная коробка.,
1,https://www.toys-land.ru/goods/kartina-pazl-re...,"Пазл-репродукция Репин И.Е - На меже, 1500 эле...",Стелла,/brand/stella,Россия,150229,1 110 ₽,собранного пазла: 85*58 см.,Количество : 1500 шт.,картон.,красочная картонная коробка.,
2,https://www.toys-land.ru/goods/pazl-germaniya-...,"Пазл Германия - Ночь в Рамзау, 500 деталей",Castorland (Касторлэнд),/brand/castorland,Польша,B-53063,360 ₽,собранного пазла: 47*33 см.,Количество : 500 шт.,картон.,красочная картонная коробка.,
3,https://www.toys-land.ru/goods/derevyannyj-paz...,"Деревянный пазл Сальвадор Дали 40*40 см, 475 э...",Active Puzzles (Актив Пазл),/brand/active-puzzles,Россия,1035-activepuzzles,4 200 ₽,собранного пазла: 40*40 см.,Количество : 475 шт.,дерево (береза).,деревянная коробка.,
4,https://www.toys-land.ru/goods/pazl-parusnaya-...,"Пазл Парусная лодка, 1000 элементов",Educa (Эдука),/brand/educa,Испания,18490,1 326 ₽,собранного пазла: 68*48 см.,Количество : 1000 шт.,картон.,красочная картонная коробка.,
...,...,...,...,...,...,...,...,...,...,...,...,...
88,https://www.toys-land.ru/goods/pazl-park-gujel...,"Пазл Парк Гуэль - Барселона, 500 элементов",Educa (Эдука),/brand/educa,Испания,15319,1 190 ₽,собранного пазла: 48*34 см.,Количество : 500 шт.,картон.,красочная картонная коробка.,
89,https://www.toys-land.ru/goods/pazl-cveta-azii...,"Пазл Цвета Азии, 1000 элементов",Educa (Эдука),/brand/educa,Испания,16294,1 380 ₽,собранного пазла: 48*68 см.,Количество : 1000 шт.,картон.,красочная картонная коробка.,
90,https://www.toys-land.ru/goods/pazl-zhestokaya...,Пазл-репродукция Жестокая преданность - Энн Ст...,Educa (Эдука),/brand/educa,Испания,17692,1 560 ₽,собранного пазла: 68*48 см.,Количество : 1000 шт.,картон.,красочная картонная коробка.,
91,https://www.toys-land.ru/goods/nabor-pazlov-za...,"Набор пазлов Загородный сад, 3*500 элементов",Educa (Эдука),/brand/educa,Испания,17965,1 777 ₽,одного пазла: 40*40.5 см.,Количество : 3*500 шт.,картон.,красочная картонная коробка.,


# puzzle wharehouse

In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from typing import List, Dict, Optional
from urllib.parse import urljoin, parse_qs, urlparse, urlencode

class PuzzleWarehouseParser:
    def __init__(self):
        self.base_url = "https://www.puzzlewarehouse.com"
        self.jigsaw_url = f"{self.base_url}/jigsaw-puzzles/"
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9,ru;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-User': '?1',
            'Cache-Control': 'max-age=0',
            'DNT': '1',
        }
        self.session.headers.update(self.headers)

    def get_page(self, url: str, retries: int = 3) -> Optional[BeautifulSoup]:
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=30)
                if response.status_code == 200:
                    return BeautifulSoup(response.content, 'html.parser')
                elif response.status_code == 403:
                    print(f"Доступ запрещен (403). Попытка {attempt + 1}/{retries}")
                    time.sleep(5 * (attempt + 1))
                else:
                    print(f"Ошибка HTTP {response.status_code}. Попытка {attempt + 1}/{retries}")
                    time.sleep(2)
            except requests.RequestException as e:
                print(f"Ошибка запроса: {e}. Попытка {attempt + 1}/{retries}")
                time.sleep(3)
        return None

    def extract_puzzle_info(self, product_thumb) -> Optional[Dict[str, str]]:
        try:
            img_tag = product_thumb.find('img')
            if not img_tag or not img_tag.get('title'):
                return None
            name = img_tag['title']

            link_tag = product_thumb.find('a')
            if not link_tag or not link_tag.get('href'):
                return None
            link = link_tag['href']
            if not link.startswith('http'):
                link = urljoin(self.base_url, link)

            parent_element = product_thumb.parent
            piece_count_element = None

            if parent_element:
                piece_count_element = parent_element.find('div', class_='piece-count')

            if not piece_count_element:
                piece_count_element = product_thumb.find_next('div', class_='piece-count')

            pieces = None
            if piece_count_element:
                piece_text = piece_count_element.get_text(strip=True)
                match = re.search(r'(\d+)\s*Pieces?', piece_text)
                if match:
                    pieces = int(match.group(1))

            if not pieces:
                match = re.search(r'(\d+)\s*[Pp]ieces?', name)
                if match:
                    pieces = int(match.group(1))

            return {
                'name': name,
                'link': link,
                'pieces': pieces if pieces else 'Unknown'
            }
        except Exception as e:
            print(f"Ошибка при извлечении информации: {e}")
            return None

    def build_page_url(self, page_number: int, params: Dict = None) -> str:
        if params is None:
            params = {
                'aid': '',
                'pcs': '',
                'cid': '',
                'sort': 'mostPopular',
                'show': '48',
                'stock': '1',
                'budget_from': '0',
                'budget_to': '0',
                'price': '0',
            }
        params['page'] = str(page_number)
        query_string = urlencode(params)
        return f"{self.jigsaw_url}?{query_string}"

    def get_total_pages(self, soup: BeautifulSoup) -> int:
        pagination = soup.find('ul', class_='pagination')
        if pagination:
            page_links = pagination.find_all('a')
            page_numbers = []
            for link in page_links:
                try:
                    num = int(link.text.strip())
                    page_numbers.append(num)
                except ValueError:
                    continue
            if page_numbers:
                return max(page_numbers)

        product_count = len(soup.find_all('div', class_='product-thumb'))
        if product_count == 0:
            return 0
        return 1

    def parse_all_pages(self, max_pages: int = None, start_page: int = 1) -> List[Dict[str, str]]:
        all_puzzles = []
        page = start_page

        if max_pages is None:
            first_url = self.build_page_url(1)
            first_soup = self.get_page(first_url)
            if first_soup:
                max_pages = self.get_total_pages(first_soup)

            else:
                max_pages = 1

        while page <= max_pages:
            url = self.build_page_url(page)


            soup = self.get_page(url)

            if not soup:
                print(f"Не удалось загрузить страницу {page}")
                page += 1
                continue

            product_thumbs = soup.find_all('div', class_='product-thumb')
            if not product_thumbs:
                print(f"На странице {page} нет товаров")
                break



            for thumb in product_thumbs:
                puzzle_info = self.extract_puzzle_info(thumb)
                if puzzle_info:
                    all_puzzles.append(puzzle_info)

            page += 1
            time.sleep(2)

        return all_puzzles

    def get_dataframe(self, max_pages: int = None, start_page: int = 1) -> pd.DataFrame:
        puzzles = self.parse_all_pages(max_pages, start_page)
        df = pd.DataFrame(puzzles, columns=['name', 'link', 'pieces'])

        return df

In [33]:
parser = PuzzleWarehouseParser()
df = parser.get_dataframe(max_pages=10)

display(df)

Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 1
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 2
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 3
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 4
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 5
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 6
Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 7
Доступ запрещен (403). Попытка 1/3
Доступ

,name,link,pieces


In [34]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from typing import List, Dict, Optional
from urllib.parse import urljoin, urlencode

class PuzzleDetailParser:
    def __init__(self):
        self.base_url = "https://www.puzzlewarehouse.com"
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9,ru;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-User': '?1',
            'Cache-Control': 'max-age=0',
            'DNT': '1',
        }
        self.session.headers.update(self.headers)

    def get_page(self, url: str, retries: int = 3) -> Optional[BeautifulSoup]:
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=30)
                if response.status_code == 200:
                    return BeautifulSoup(response.content, 'html.parser')
                elif response.status_code == 403:
                    print(f"Доступ запрещен (403). Попытка {attempt + 1}/{retries}")
                    time.sleep(5 * (attempt + 1))
                else:
                    print(f"Ошибка HTTP {response.status_code}. Попытка {attempt + 1}/{retries}")
                    time.sleep(2)
            except requests.RequestException as e:
                print(f"Ошибка запроса: {e}. Попытка {attempt + 1}/{retries}")
                time.sleep(3)
        return None

    def extract_price(self, soup: BeautifulSoup) -> Optional[str]:
        price_elem = soup.find('span', class_='product-unit-price')
        if price_elem:
            return price_elem.text.strip()

        price_div = soup.find('div', class_='product-price')
        if price_div:
            span = price_div.find('span')
            if span:
                return span.text.strip()
        return None

    def extract_theme(self, soup: BeautifulSoup) -> Optional[str]:
        strong_tags = soup.find_all('strong')
        for strong in strong_tags:
            if 'Themes:' in strong.text or 'Theme:' in strong.text:
                link = strong.find('a')
                if link:
                    return link.text.strip()
                next_sibling = strong.next_sibling
                if next_sibling:
                    text = str(next_sibling).strip()
                    if text:
                        return text
        return None

    def extract_age(self, soup: BeautifulSoup) -> Optional[str]:
        age_td = soup.find('td', string=re.compile(r'^\s*Age\s*$'))
        if age_td:
            age_row = age_td.find_parent('tr')
            if age_row:
                cells = age_row.find_all('td')
                if len(cells) >= 2:
                    return cells[1].get_text(strip=True)

        tbody = soup.find('tbody')
        if tbody:
            rows = tbody.find_all('tr')
            for row in rows:
                cells = row.find_all('td')
                if len(cells) >= 2:
                    key = cells[0].get_text(strip=True)
                    if key == 'Age':
                        return cells[1].get_text(strip=True)
        return None

    def extract_puzzle_category(self, soup: BeautifulSoup) -> Optional[str]:
        category_td = soup.find('td', string=re.compile(r'^\s*Puzzle Category\s*$'))
        if category_td:
            category_row = category_td.find_parent('tr')
            if category_row:
                cells = category_row.find_all('td')
                if len(cells) >= 2:
                    return cells[1].get_text(strip=True)

        tbody = soup.find('tbody')
        if tbody:
            rows = tbody.find_all('tr')
            for row in rows:
                cells = row.find_all('td')
                if len(cells) >= 2:
                    key = cells[0].get_text(strip=True)
                    if key == 'Puzzle Category':
                        return cells[1].get_text(strip=True)
        return None

    def parse_puzzle_details(self, url: str) -> Optional[Dict[str, str]]:
        soup = self.get_page(url)

        if not soup:
            print(f"Не удалось загрузить страницу: {url}")
            return None

        title_elem = soup.find('h1')
        name = title_elem.text.strip() if title_elem else None

        price = self.extract_price(soup)
        theme = self.extract_theme(soup)
        age = self.extract_age(soup)
        puzzle_category = self.extract_puzzle_category(soup)

        piece_count_elem = soup.find('div', class_='piece-count')
        pieces = None
        if piece_count_elem:
            piece_text = piece_count_elem.get_text(strip=True)
            match = re.search(r'(\d+)', piece_text)
            if match:
                pieces = int(match.group(1))

        result = {
            'name': name,
            'url': url,
            'price': price,
            'pieces': pieces if pieces else 'Unknown',
            'theme': theme,
            'age': age,
            'puzzle_category': puzzle_category
        }

        return result

    def parse_multiple_puzzles(self, urls: List[str]) -> pd.DataFrame:
        results = []
        for url in urls:
            puzzle_data = self.parse_puzzle_details(url)
            if puzzle_data:
                results.append(puzzle_data)

            time.sleep(1)

        df = pd.DataFrame(results)

        return df


class PuzzleWarehouseFullParser:
    def __init__(self):
        self.base_url = "https://www.puzzlewarehouse.com"
        self.jigsaw_url = f"{self.base_url}/jigsaw-puzzles/"
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9,ru;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-User': '?1',
            'Cache-Control': 'max-age=0',
            'DNT': '1',
        }
        self.session.headers.update(self.headers)
        self.detail_parser = PuzzleDetailParser()
        self.detail_parser.session = self.session

    def get_page(self, url: str, retries: int = 3) -> Optional[BeautifulSoup]:
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=30)
                if response.status_code == 200:
                    return BeautifulSoup(response.content, 'html.parser')
                elif response.status_code == 403:
                    print(f"Доступ запрещен (403). Попытка {attempt + 1}/{retries}")
                    time.sleep(5 * (attempt + 1))
                else:
                    print(f"Ошибка HTTP {response.status_code}. Попытка {attempt + 1}/{retries}")
                    time.sleep(2)
            except requests.RequestException as e:
                print(f"Ошибка запроса: {e}. Попытка {attempt + 1}/{retries}")
                time.sleep(3)
        return None

    def extract_product_links(self, soup: BeautifulSoup) -> List[str]:
        links = []
        product_thumbs = soup.find_all('div', class_='product-thumb')
        for thumb in product_thumbs:
            link_tag = thumb.find('a')
            if link_tag and link_tag.get('href'):
                link = link_tag['href']
                if not link.startswith('http'):
                    link = urljoin(self.base_url, link)
                links.append(link)
        return links

    def build_page_url(self, page_number: int) -> str:
        params = {
            'aid': '',
            'pcs': '',
            'cid': '',
            'sort': 'mostPopular',
            'show': '48',
            'stock': '1',
            'budget_from': '0',
            'budget_to': '0',
            'price': '0',
            'page': str(page_number)
        }
        return f"{self.jigsaw_url}?{urlencode(params)}"

    def parse_catalog_with_details(self, max_pages: int = 1, start_page: int = 1) -> pd.DataFrame:
        all_links = []

        for page in range(start_page, start_page + max_pages):
            url = self.build_page_url(page)


            soup = self.get_page(url)
            if not soup:
                print(f"Не удалось загрузить страницу {page}")
                break

            links = self.extract_product_links(soup)
            if not links:
                print(f"На странице {page} нет товаров")
                break


            all_links.extend(links)
            time.sleep(1)



        df = self.detail_parser.parse_multiple_puzzles(all_links)
        return df



In [35]:
parser = PuzzleWarehouseFullParser()
df = parser.parse_catalog_with_details(max_pages=1)
display(df)

Доступ запрещен (403). Попытка 1/3
Доступ запрещен (403). Попытка 2/3
Доступ запрещен (403). Попытка 3/3
Не удалось загрузить страницу 1


""
